<a href="https://colab.research.google.com/github/josegoms/testing/blob/main/chocolate_sales_performance_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3 - Dataset Overview

- Source: Kaggle Datasets
- Rows: 3283 (Including headers)
- Columns:
    - Date - Transaction date (DD/MM/YYYY Format)
    - Product - Full product name
    - Amount - Sales amount
    - Boxes - Quantity of boxes in each transaction
    - Country  - Customer’s country
    - Sales Person - name of the sales representative

# 4 - Data Preparation and Cleaning
Creating the environment and loading key libraries like Pandas and Numpy.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
print('Sucessful load')

/kaggle/input/chocolate-sales/ChocolateSales.pbix
/kaggle/input/chocolate-sales/Chocolate Sales (2).csv
/kaggle/input/chocolate-sales/ChocolateSales.pdf
/kaggle/input/chocolate-sales/ChocolateSales.pbit
/kaggle/input/datasets/josgomespinheironeto/chocolate-sales-new/Chocolate Sales (2) new.csv
Sucessful load


## 4.1 - Data Inspection
Basic evaluation of the overall dataset, with the goal of cleaning and managing null cells and duplicated values. First tool used is Excel to, by eye, spot critical errors that could prevent a better analysis forward. First task was to properly format the data since it's a CSV file and opens on Excel with all data glued on the first column. For this, I used the Import Text Wizard, tool responsible for reading data in other formats, including `.csv`, and transforming it to Excel formats, as `.xls` or `.xlsx`.
* Column `Sales Person` has data type of string/text.
* Column `Country` has data type of string/text.
* Column `Product` has data of string/text.
* Column `Date` has data type of datetime. Probably changed automatically by Import Text Wizard.
* Column `Amount` has data type of string/text. Necessary change to Numeric/Integer/Float type.
* Column `Boxes Shipped` has data type of string/text. Necessary change to Numeric/Integer/Float type.

No missing values, spelling errors, or duplicated values found in a first Excel analysis.

## 4.2 - Transformations
1. Transform the Date column data type into actual datetime64 data type.

In [ ]:
# Load dataset
db = pd.read_csv('/kaggle/input/chocolate-sales/Chocolate Sales (2).csv')

# Update column with new data type
db['Date'] = pd.to_datetime(db['Date'], dayfirst=True)

2. Change for a cleaner analysis is to add a columns that calculates the individual price of each box in each transaction. It helps to have a macro and micro vision of the data, also highlights sales person who sells low price products compare to a sales person versed in selling high price products.

In [ ]:
#Remove non-numeric character from Amount to change it from str to int
def remove(cell):
    return cell.replace('$', '').replace(',', '')

# Turn into numeric
db['Amount'] = pd.to_numeric(db['Amount'].map(remove))

# Function to calculate revenue per box
def calculate(row):
    return row['Amount'] / row['Boxes Shipped']

# Read every row and store new Series
new_column = db.apply(calculate, axis='columns')

# Round and assign it
db['Revenue per Box'] = new_column.round(0)

3. Now, let's create a column that extract only the month of each transaction. It's effective to use later as a base to analysis on periods of time with the most transactions and spot fluctuations trends that could connect with specific holidays.

In [ ]:
# Extract each transacation month and assign to a new column
db['Month'] = db['Date'].dt.month
db.to_csv('choco_sales.csv', index=False)

# 5. Key Performance Indicators (KPIs)
In this section, we're going turn data into information by using SQL queries.

* Total Revenue → Measures overall business scale.

In [ ]:
query = """SELECT SUM(Amount) AS Total_Revenue
            FROM choco_sales;
        """

| Total_Revenue |
| --- |
| 1303402410 |


* Revenue by Country → Identifies geographic contribution.

In [ ]:
query = """SELECT country, SUM("Boxes Shipped") AS Total_boxes, SUM(Amount) AS Revenue_per_Country,
           SUM(Amount)/SUM("Boxes Shipped") AS Avg_price_per_box
           FROM choco_sales
           GROUP BY country
		   ORDER BY Revenue_per_Country DESC, Avg_price_per_box DESC;
        """

| Country      | Total_boxes | Revenue_per_Country | Avg_price_per_box |
|-------------|------------|--------------------|-------------------|
| Australia   | 99618      | 243278733          | 2442              |
| USA         | 81820      | 220385614          | 2693              |
| UK          | 92523      | 219184578          | 2368              |
| India       | 89968      | 216005965          | 2400              |
| Canada      | 95158      | 206129450          | 2166              |
| New Zealand | 81350      | 198418070          | 2439              |

* Revenue by Sales Person → Measures individual productivity.

In [ ]:
query = """SELECT "Sales Person", SUM(Amount) AS Revenue_per_Sales_Person, SUM("Boxes Shipped") AS Total_boxes,
           SUM(Amount)/SUM("Boxes Shipped") AS Avg_price_per_box
           FROM choco_sales
           GROUP BY "Sales Person"
		   ORDER BY Revenue_per_Sales_Person DESC, Avg_price_per_box DESC;
        """

| Sales Person           | Revenue_per_Sales_Person | Total_boxes | Avg_price_per_box |
|------------------------|--------------------------|------------|-------------------|
| Kelci Walkden          | 69371340                 | 26605      | 2607              |
| Brien Boise            | 69289878                 | 24738      | 2800              |
| Madelene Upcott        | 68195064                 | 22199      | 3071              |
| Ches Bonnell           | 66934753                 | 23070      | 2901              |
| Van Tuxwell            | 66078076                 | 20627      | 3203              |
| Oby Sorrel             | 65921004                 | 26390      | 2497              |
| Dennison Crosswaite    | 62715934                 | 26862      | 2334              |
| Beverie Moffet         | 59928174                 | 28027      | 2138              |
| Barr Faughny           | 57552319                 | 19520      | 2948              |
| Marney O'Breen         | 57130545                 | 24595      | 2322              |
| Roddy Speechley        | 51456808                 | 21130      | 2435              |
| Kaine Padly            | 51045612                 | 22134      | 2306              |
| Gunar Cockshoot        | 50690897                 | 20299      | 2497              |
| Curtice Advani         | 46968737                 | 21599      | 2174              |
| Jan Morforth           | 46798479                 | 23360      | 2003              |
| Karlen McCaffrey       | 46553830                 | 29553      | 1575              |
| Jehu Rudeforth         | 45904105                 | 22104      | 2076              |
| Gigi Bohling           | 44200152                 | 19237      | 2297              |
| Mallorie Waber         | 43479543                 | 18219      | 2386              |
| Andria Kimpton         | 42899488                 | 19730      | 2174              |
| Camilla Castle         | 42426831                 | 16505      | 2570              |
| Dotty Strutley         | 40790499                 | 20927      | 1949              |
| Husein Augar           | 40274267                 | 17683      | 2277              |
| Rafaelita Blaksland    | 37767585                 | 13091      | 2885              |
| Wilone O'Kielt         | 29028490                 | 12233      | 2372              |


* Revenue by Product → Evaluates product demand.

In [ ]:
query = """SELECT Product, SUM("Boxes Shipped") AS Total_boxes, SUM(Amount) AS Revenue_per_Product,
           SUM(Amount)/SUM("Boxes Shipped") AS Avg_price_per_box
           FROM choco_sales
           GROUP BY Product
		   ORDER BY Revenue_per_Product DESC, Avg_price_per_box DESC;
        """

| Product                 | Total_boxes | Revenue_per_Product | Avg_price_per_box |
|--------------------------|------------|---------------------|-------------------|
| Smooth Sliky Salty       | 26969      | 73619017            | 2729              |
| 50% Dark Bites           | 29810      | 72837584            | 2443              |
| Peanut Butter Cubes      | 25339      | 69795637            | 2754              |
| White Choc               | 25158      | 67353846            | 2677              |
| Eclairs                  | 26678      | 65928209            | 2471              |
| 85% Dark Bars            | 23828      | 62972528            | 2642              |
| Organic Choco Syrup      | 23602      | 61259575            | 2595              |
| Spicy Special Slims      | 26662      | 61047436            | 2289              |
| Manuka Honey Choco       | 23736      | 60521659            | 2549              |
| Mint Chip Choco          | 25149      | 60016855            | 2386              |
| 99% Dark & Pure          | 24818      | 59596797            | 2401              |
| Almond Choco             | 20558      | 57781283            | 2810              |
| After Nines              | 25156      | 57301671            | 2277              |
| Drinking Coco            | 26402      | 56930678            | 2156              |
| Raspberry Choco          | 21672      | 56463233            | 2605              |
| Milk Bars                | 25436      | 56086861            | 2205              |
| Fruit & Nut Bars         | 23632      | 55320216            | 2340              |
| Baker's Choco Chips      | 21448      | 54549285            | 2543              |
| Orange Choco             | 23607      | 54511459            | 2309              |
| Choco Coated Almonds     | 19677      | 50620121            | 2572              |
| Caramel Stuffed Bars     | 26576      | 46122079            | 1735              |
| 70% Dark Bites           | 24524      | 42766381            | 1743              |


* Revenue per Box → Assesses pricing efficiency.

In [ ]:
query = """SELECT AVG("Revenue per Box") AS Average_per_Box
            FROM choco_sales;
        """

| Average_per_Box |
| --- |
| 1113.40950639854 |

* Monthly Revenue → Detects seasonality.

In [ ]:
query = """SELECT Month, SUM("Boxes Shipped") AS Total_boxes, SUM(Amount) AS Revenue_per_Month,
           SUM(Amount)/SUM("Boxes Shipped") AS Avg_price_per_box
           FROM choco_sales
           GROUP BY Month
		   ORDER BY Revenue_per_Month DESC, Avg_price_per_box;
        """

| Month | Total_boxes | Revenue_per_Month | Avg_price_per_box |
|-------|------------|-------------------|-------------------|
| 1     | 84162      | 189298173         | 2249              |
| 6     | 80357      | 181271250         | 2255              |
| 7     | 69808      | 170393846         | 2440              |
| 3     | 59633      | 159835390         | 2680              |
| 5     | 66662      | 159225336         | 2388              |
| 8     | 60682      | 149989744         | 2471              |
| 2     | 54917      | 149913303         | 2729              |
| 4     | 64216      | 143475368         | 2234              |

# 6 - Performance Analysis